## Predictive Machine Learning Modeling

---
## Step 1 - Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score, confusion_matrix
from sklearn.calibration import calibration_curve
import statsmodels.api as sm
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
%matplotlib inline

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

TITLE_TEAMS = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']

TEAM_PALETTE = {
    'Arsenal':            '#EF0107',
    'Liverpool':          '#00B2A9',
    'Manchester City':    '#6CABDD',
    'Manchester United':  '#FFB81C',
}

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print('Imports ready')

---
## Step 2 - Load Data

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df['season'] = df['season'].astype(str)
df = df.sort_values(['team', 'date']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Teams: {sorted(df["team"].unique())}')

---
## Section A - Target Formulation

In [ ]:
# is_home from venue, y from result (L/D/W -> 0/1/2)
# class balance check, this is the accuracy floor to beat later


### Reading Section A

---
## Section B - Feature Matrix, Part 1: Existing Features

In [ ]:
# recap: xG_roll5, xGA_roll5, pts_roll5, win_rate_roll5, stakes_intensity,
# is_home, is_rivalry (adjusted in Section D)
# do NOT include is_high_stakes / is_high_stakes_retro as separate features,
# stakes_intensity is the continuous version and retro leaks future info


---
## Section C - Feature Matrix, Part 2: Opponent Quality

In [ ]:
# reload full 20-team Understat schedule from local cache (sd.Understat),
# same trick NB02 used for title_gap, no new scraping


In [ ]:
# wide-to-long reshape, this time keeping home_xg/away_xg as xG/xGA


In [ ]:
# opponent's own rolling xG/xGA, shift(1) before rolling, same anti-leakage
# rule as NB02's own rolling features


In [ ]:
# merge onto main df by game_id + opponent, build opp_xgd_roll5
# check null count, should only be the same early-season gap as our own
# rolling features


In [ ]:
# spot check one match by hand before trusting the merge at scale


### Reading Section C

---
## Section D - Big 6 vs Rivalry (fixing a double-count)

In [ ]:
# is_big6_opp from a fixed Big 6 list
# check overlap between is_rivalry and is_big6_opp before deciding anything


In [ ]:
# is_non_big6_rivalry = is_rivalry AND NOT is_big6_opp
# confirm zero overlap with is_big6_opp


### Reading Section D

---
## Section E - Assembling the Feature Matrix

In [ ]:
# FEATURES list, drop rows with NaN rolling features
# recency_weight goes in as sample_weight at fit time, NOT as an X column


---
## Section F - Time-Series Train/Test Split

In [ ]:
# headline split: train seasons <= 2024-25, test = 2025-26


In [ ]:
# walk-forward stability check: expanding window across season boundaries


### Reading Section F

---
## Section G - Baseline: Dummy Classifier

In [ ]:
# DummyClassifier(strategy='prior'), log loss + accuracy floor


---
## Section H - Multinomial Logistic Regression

In [ ]:
# StandardScaler, sklearn LogisticRegression, log loss / accuracy vs dummy


In [ ]:
# statsmodels MNLogit for coefficients + p-values, isolate stakes_intensity


In [ ]:
# confusion matrix


### Reading Section H

---
## Section I - Calibration

In [ ]:
# calibration_curve for the Win class, reliability diagram


### Reading Section I

---
## Section J - Random Forest

In [ ]:
# RandomForestClassifier, conservative depth given ~1000 rows
# log loss / accuracy vs logistic regression and dummy


---
## Section K - XGBoost

In [ ]:
# XGBClassifier, conservative hyperparameters (shallow, regularized)


In [ ]:
# summary table: dummy / logreg / rf / xgb -> log loss, accuracy, brier score


In [ ]:
# walk-forward consistency check: does logistic regression beat the
# same-season dummy in every fold, not just the headline split


### Reading Section K

---
## Section L - SHAP Values

In [ ]:
# TreeExplainer on the XGBoost model, mean |SHAP value| per feature
# for the Win class


In [ ]:
# feature importance bar chart


### Reading Section L

---
## Key Findings Summary